In [2]:
#!/usr/bin/env Rscript

# =============================================================================
# MDS_index_profiles_OHFP_compute_v2_correctedDist_isoStress.R
# -----------------------------------------------------------------------------
# Purpose:
#   Compute ONLY:
#     - classical MDS (cmdscale)  &  non-metric MDS (isoMDS)
#     - For each:
#         method_tag ∈ {classical, nonmetric}
#         U_tag      ∈ {U10, U25, U50}  (max k)
#         dataset    ∈ {OH, FP}
#         mode       ∈ {top3, cumeig}
#       → Ward.D2 (Euclidean) clustering for k = 2..K_max
#       → indices: silhouette, dunn, gap, ch, db, ptbiserial
#
# Distance:
#   - Pairwise correlation: Corr_ij (use = "pairwise.complete.obs")
#   - Raw distance:        d'_ij = 1 - Corr_ij
#   - Effective samples:   n_ij  = #rows where Xi, Xj are both non-NA
#   - Corrected distance:  d_ij  = (M / n_ij) * d'_ij  (M = total #samples)
#   - Clamped to [0, 2]; NA -> 2
#
# isoMDS stress:
#   - For nonmetric MDS, final stress is saved as:
#       <out_root>/isoMDS_stress/isoMDS_stress_{dataset}_{mode}_{method}_{TS}.csv
#
# Inputs:
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/
#     - preprocessed_features_OH.csv
#     - preprocessed_features_FP.csv
#
# Outputs (CSV only; NO figures here):
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/OH_FP_only/
#     {method_tag}/{U_tag}/{dataset}/csv_profiles/
#       - index_profile_{mode}_{index}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#       - index_profiles_all_{dataset}_{mode}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#
# =============================================================================

# ==== CONFIG (edit here) =====================================================

base_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"

# Input feature files (OH/FP) for MDS
data_dir <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "data",
  "for_MDS_STEP2"
)

# Root of all outputs for this script
out_root <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "4.2_clustering",
  "OH_FP_only"
)

# Max cluster numbers: k = 2..K_max for each tag
max_k_list <- c(U10 = 10, U25 = 25, U50 = 50)

# MDS methods
method_tags <- c("classical", "nonmetric")

# Gap statistic bootstrap iterations
gap_B <- 20

# Input feature file names
dataset_files <- c(
  OH = "preprocessed_features_OH.csv",
  FP = "preprocessed_features_FP.csv"
)

# Dimension selection modes
dim_modes   <- c("top3", "cumeig")

# Indices to compute
index_names <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")

# Timestamp for this compute run
ts_str <- format(Sys.time(), "%Y%m%d_%H%M%S")

# =============================================================================
# Libraries & basic helpers
# =============================================================================

safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("readr")
safe_lib("cluster")  # silhouette, clusGap
safe_lib("fpc")      # dunn via cluster.stats
safe_lib("MASS")     # isoMDS

set.seed(42)

ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",",
                   row.names = 1, as.is = TRUE, strip.white = FALSE)
  is_num <- !vapply(df, is.character, logical(1))
  if (!any(is_num)) stop("No numeric columns: ", path)
  df[, is_num, drop = FALSE]
}

drop_zero_var_cols <- function(M){
  M <- as.matrix(M)
  keep <- apply(M, 2, function(v) sd(v, na.rm=TRUE) > 0)
  if (!any(keep)) stop("All columns have zero variance.")
  M[, keep, drop=FALSE]
}

# =============================================================================
# MDS (classical) from correlation + "missingness-corrected" distance
#   - Step1: pairwise correlation (use = "pairwise.complete.obs")
#   - Step2: effective sample size n_ij for each pair (i, j)
#   - Step3: raw distance d'_ij = 1 - Corr_ij
#   - Step4: corrected distance d_ij = (M / n_ij) * d'_ij  (M = total samples)
#   - Step5: clamp to [0, 2] and convert to dist object
# =============================================================================

compute_mds_and_dist <- function(numData, k_max_eig = 50){
  # numData: rows = samples (M), cols = variables (P)
  numData <- as.matrix(numData)
  M <- nrow(numData)
  P <- ncol(numData)

  # --- pairwise correlation (eq. 4.1 base) ---
  # Corr(X_i, X_j) using only samples where both variables are observed.
  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))

  # --- effective sample size for each pair (i, j): n_ij ---
  # n_ij = number of rows where BOTH X_i and X_j are non-NA.
  valid   <- !is.na(numData)   # M x P logical
  n_eff   <- crossprod(valid)  # P x P numeric

  # --- raw distance from correlation: d'_ij = 1 - Corr_ij ---
  d_raw <- 1 - corData

  # --- corrected distance: d_ij = (M / n_ij) * d'_ij ---
  d_corr <- matrix(NA_real_, nrow = P, ncol = P)
  dimnames(d_corr) <- dimnames(corData)

  ok <- (!is.na(d_raw)) & (n_eff > 1)
  d_corr[ok] <- (M / n_eff[ok]) * d_raw[ok]

  # diagonals should be zero distance
  diag(d_corr) <- 0

  # --- clamp distances to [0, 2] ---
  d_corr[d_corr < 0] <- 0
  d_corr[d_corr > 2] <- 2

  # any remaining NA (e.g., n_eff <= 1) → set to max distance (2)
  d_corr[is.na(d_corr)] <- 2

  # convert to "dist" object for cmdscale / hclust
  D <- as.dist(d_corr)

  # --- classical MDS (cmdscale) ---
  # cmdscale() internally performs Gower (1971) double-centering and
  # eigen-decomposition corresponding to eqs. (4.3)–(4.5).
  k <- min(k_max_eig, max(1, P - 1))
  mds <- cmdscale(D, k = k, eig = TRUE)

  # eigenvalue contributions (for top3 / cumeig selection)
  eig_vals <- mds$eig
  totaleig <- sum(abs(eig_vals))
  peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else abs(eig_vals) / totaleig

  contrib <- data.frame(
    axis        = seq_along(eig_vals),
    eigenvalue  = eig_vals,
    percent     = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )

  list(D = D, mds = mds, contrib = contrib)
}

select_dims <- function(contrib_df, mode, avail_dims){
  pos_idx <- which(contrib_df$percent > 0)
  if (length(pos_idx) == 0) return(min(3, max(1, avail_dims)))
  if (mode == "top3"){
    k <- min(3, length(pos_idx))
  } else {
    thr <- 0.8
    k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
    if (is.na(k)) k <- length(pos_idx)
  }
  k <- min(k, avail_dims); k <- max(1, k)
  k
}

# =============================================================================
# Index implementations (CH / DB / point-biserial)
# =============================================================================

compute_db_index <- function(X, cl){
  if (length(unique(cl)) < 2) return(NA_real_)
  X <- as.matrix(X)
  labs <- sort(unique(cl))
  k <- length(labs)

  cent <- sapply(labs, function(L){ colMeans(X[cl == L, , drop = FALSE]) })
  cent <- t(cent)

  S <- numeric(k)
  for (i in seq_len(k)){
    idx <- which(cl == labs[i])
    if (length(idx) <= 1){
      S[i] <- 0
    } else {
      ci <- cent[i, , drop = FALSE]
      S[i] <- mean(sqrt(rowSums((X[idx, , drop = FALSE] -
                                   matrix(ci, nrow=length(idx), ncol=ncol(X), byrow = TRUE))^2)))
    }
  }

  Dc <- as.matrix(dist(cent))
  R <- matrix(NA_real_, k, k)
  for (i in seq_len(k)){
    for (j in seq_len(k)){
      if (i == j) next
      if (Dc[i, j] == 0) {
        R[i, j] <- Inf
      } else {
        R[i, j] <- (S[i] + S[j]) / Dc[i, j]
      }
    }
  }
  db <- mean(apply(R, 1, function(v) max(v[is.finite(v)], na.rm = TRUE)), na.rm = TRUE)
  if (!is.finite(db)) return(NA_real_)
  db
}

compute_ptbiserial_index <- function(D, cl){
  n <- attr(D, "Size")
  if (length(unique(cl)) < 2 || n < 2) return(NA_real_)
  s <- logical(n * (n - 1) / 2)
  idx <- 1
  for (i in 1:(n-1)){
    for (j in (i+1):n){
      s[idx] <- (cl[i] == cl[j]); idx <- idx + 1
    }
  }
  d <- as.numeric(D)
  if (length(unique(s)) < 2) return(NA_real_)
  r <- suppressWarnings(cor(d, as.numeric(s), method = "pearson", use = "complete.obs"))
  if (is.na(r)) return(NA_real_)
  -r
}

compute_ch_index <- function(X, cl){
  X <- as.matrix(X)
  n <- nrow(X)
  labs <- sort(unique(cl))
  k <- length(labs)
  if (k < 2 || n <= k) return(NA_real_)

  m_all <- colMeans(X)
  ssb <- 0
  ssw <- 0
  for (g in labs){
    idx <- which(cl == g)
    Xg  <- X[idx, , drop = FALSE]
    mg  <- colMeans(Xg)
    ssb <- ssb + nrow(Xg) * sum((mg - m_all)^2)
    ssw <- ssw + sum(rowSums((Xg - matrix(mg, nrow = nrow(Xg),
                                          ncol = ncol(Xg), byrow = TRUE))^2))
  }
  if (ssw <= 0) return(NA_real_)
  (ssb / (k - 1)) / (ssw / (n - k))
}

# =============================================================================
# Get MDS coordinates (classical / nonmetric) + isoMDS stress save
# =============================================================================

get_X_for <- function(dataset_tag = c("OH","FP"),
                      mode = c("top3","cumeig"),
                      method_tag = c("classical","nonmetric")){
  dataset_tag <- match.arg(dataset_tag)
  mode        <- match.arg(mode)
  method_tag  <- match.arg(method_tag)

  csv_name <- dataset_files[[dataset_tag]]
  in_csv <- file.path(data_dir, csv_name)
  if (!file.exists(in_csv)) stop("Input file not found: ", in_csv)

  message(sprintf("\n[Info] Reading features: %s (%s)", in_csv, dataset_tag))
  numData <- read_numeric_matrix(in_csv)

  md <- compute_mds_and_dist(numData)
  D   <- md$D
  mds <- md$mds
  contrib <- md$contrib

  coords_classical <- as.matrix(mds$points)
  if (is.null(dim(coords_classical))) {
    coords_classical <- matrix(coords_classical, ncol = 1)
    colnames(coords_classical) <- "Dim1"
  }

  k_dim <- select_dims(contrib, mode, ncol(coords_classical))
  message(sprintf("[Info] %s / %s: selected dim = %d (method=%s)",
                  dataset_tag, mode, k_dim, method_tag))

  if (method_tag == "classical"){
    X <- coords_classical[, 1:k_dim, drop = FALSE]
  } else {
    # non-metric MDS (isoMDS) using corrected distance D
    init <- coords_classical[, 1:k_dim, drop = FALSE]
    iso <- try(
      MASS::isoMDS(D, y = init, k = k_dim, maxit = 50, trace = TRUE),
      silent = TRUE
    )
    if (inherits(iso, "try-error")){
      message("  [Warn] isoMDS failed → fall back to classical coords")
      X <- init
    } else {
      stress_val <- iso$stress
      message(sprintf("  [Info] isoMDS final stress: %.4f", stress_val))

      # ---- save isoMDS stress to CSV ----
      # <out_root>/isoMDS_stress/isoMDS_stress_{dataset}_{mode}_{method}_{TS}.csv
      stress_dir <- file.path(out_root, "isoMDS_stress")
      ensure_dir(stress_dir)
      out_f <- file.path(
        stress_dir,
        sprintf("isoMDS_stress_%s_%s_%s_%s.csv",
                dataset_tag, mode, method_tag, ts_str)
      )
      df_stress <- data.frame(
        dataset = dataset_tag,
        mode    = mode,
        method  = method_tag,
        stress  = stress_val,
        stringsAsFactors = FALSE
      )
      readr::write_csv(df_stress, out_f)
      message("[Saved isoMDS stress] ", out_f)

      X <- iso$points
    }
  }

  X <- scale(drop_zero_var_cols(X))
  list(X = X, D = dist(X))
}

# =============================================================================
# Index profile computation over k = 2..K_max
# =============================================================================

ward_cut <- function(x, k){
  cl <- cutree(hclust(dist(x), method = "ward.D2"), k)
  list(cluster = cl)
}

compute_index_profiles <- function(
  X,
  D,
  dataset_tag,
  mode_tag,
  U_tag,
  k_max,
  out_csv_dir,
  method_tag
){
  ensure_dir(out_csv_dir)

  n <- nrow(X)
  k_max <- min(k_max, max(2, n - 1))
  ks <- 2:k_max

  message(sprintf("\n=== [Compute] method=%s, dataset=%s, mode=%s, U=%s (k=2..%d) ===",
                  method_tag, dataset_tag, mode_tag, U_tag, k_max))

  # ---- GAP (cluster::clusGap) ----
  gap_vec <- rep(NA_real_, length(ks))
  if (n >= 3) {
    set.seed(42)
    gap_obj <- try(
      cluster::clusGap(X, FUNcluster = ward_cut,
                       K.max = k_max, B = gap_B, verbose = FALSE),
      silent = TRUE
    )
    if (!inherits(gap_obj, "try-error")) {
      gap_tab <- as.data.frame(gap_obj$Tab)
      gap_tab$k <- seq_len(nrow(gap_tab))
      gap_tab <- gap_tab[gap_tab$k %in% ks, c("k","gap")]
      for (j in seq_len(nrow(gap_tab))) {
        pos <- which(ks == gap_tab$k[j])
        if (length(pos) == 1) gap_vec[pos] <- gap_tab$gap[j]
      }
    } else {
      warning("clusGap failed for ", dataset_tag, "/", mode_tag, "/", U_tag,
              " (method=", method_tag, "): ",
              conditionMessage(attr(gap_obj, "condition")))
    }
  }

  # ---- other indices over k ----
  sil  <- dunn <- db <- ch <- ptb <- rep(NA_real_, length(ks))

  hc <- hclust(D, method = "ward.D2")

  for (i in seq_along(ks)){
    k <- ks[i]
    cl_raw <- cutree(hc, k)
    cl <- as.integer(factor(cl_raw))
    k_eff <- length(unique(cl))
    sizes <- sort(table(cl))

    message(sprintf("\n[DEBUG] %s/%s/%s/%s k=%d (eff=%d), sizes: %s",
                    method_tag, dataset_tag, mode_tag, U_tag,
                    k, k_eff, paste(sizes, collapse = ",")))

    if (k_eff < 2){
      message("  -> k_eff < 2, all indices set to NA")
      next
    }

    # silhouette
    sil[i] <- tryCatch({
      sil_obj <- cluster::silhouette(cl, D)
      val <- mean(sil_obj[,3], na.rm = TRUE)
      message(sprintf("  silhouette = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR silhouette] %s", conditionMessage(e)))
      NA_real_
    })

    # dunn (via fpc::cluster.stats)
    dstat <- try(fpc::cluster.stats(D, cl), silent = TRUE)
    if (!inherits(dstat, "try-error") && !is.null(dstat$dunn)){
      dunn[i] <- dstat$dunn
      message(sprintf("  dunn = %s",
                      ifelse(is.na(dunn[i]), "NA", sprintf("%.6f", dunn[i]))))
    } else if (inherits(dstat, "try-error")){
      message(sprintf("  [ERROR dunn] %s",
                      conditionMessage(attr(dstat, "condition"))))
    }

    # DB
    db[i] <- tryCatch({
      val <- compute_db_index(X, cl)
      message(sprintf("  DB = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR DB] %s", conditionMessage(e)))
      NA_real_
    })

    # CH
    ch[i] <- tryCatch({
      val <- compute_ch_index(X, cl)
      message(sprintf("  CH = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR CH] %s", conditionMessage(e)))
      NA_real_
    })

    # ptbiserial
    ptb[i] <- tryCatch({
      val <- compute_ptbiserial_index(D, cl)
      message(sprintf("  ptbiserial = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR ptbiserial] %s", conditionMessage(e)))
      NA_real_
    })
  }

  # ---- save per-index CSVs ----
  profiles <- list(
    silhouette = data.frame(k = ks, score = sil),
    dunn       = data.frame(k = ks, score = dunn),
    gap        = data.frame(k = ks, score = gap_vec),
    ch         = data.frame(k = ks, score = ch),
    db         = data.frame(k = ks, score = db),
    ptbiserial = data.frame(k = ks, score = ptb)
  )

  for (ix in names(profiles)){
    df <- profiles[[ix]]
    out_f <- file.path(
      out_csv_dir,
      sprintf("index_profile_%s_%s_%s_%s_kmax%d_%s.csv",
              mode_tag, ix, method_tag, U_tag, k_max, ts_str)
    )
    readr::write_csv(df, out_f)
    message("[Saved] ", out_f)
  }

  # ---- save combined all-k CSV ----
  all_k_df <- data.frame(
    k          = ks,
    silhouette = sil,
    dunn       = dunn,
    gap        = gap_vec,
    ch         = ch,
    db         = db,
    ptbiserial = ptb,
    stringsAsFactors = FALSE
  )
  out_all <- file.path(
    out_csv_dir,
    sprintf("index_profiles_all_%s_%s_%s_%s_kmax%d_%s.csv",
            dataset_tag, mode_tag, method_tag, U_tag, k_max, ts_str)
  )
  readr::write_csv(all_k_df, out_all)
  message("[Saved all-k indices] ", out_all)

  invisible(NULL)
}

# =============================================================================
# MAIN
# =============================================================================

ensure_dir(out_root)

for (method_tag in method_tags){
  message(sprintf("\n================ METHOD = %s ================\n", method_tag))
  method_root <- file.path(out_root, method_tag)
  ensure_dir(method_root)

  for (U_tag in names(max_k_list)){
    k_max <- max_k_list[[U_tag]]
    message(sprintf("\n######## U-tag = %s (k_max = %d) ########", U_tag, k_max))

    out_U_root <- file.path(method_root, U_tag)
    ensure_dir(out_U_root)

    for (dataset_tag in names(dataset_files)){
      out_ds_root <- file.path(out_U_root, dataset_tag)
      out_ds_csv  <- file.path(out_ds_root, "csv_profiles")
      ensure_dir(out_ds_root); ensure_dir(out_ds_csv)

      for (mode_tag in dim_modes){

        mdx <- get_X_for(dataset_tag, mode_tag, method_tag)
        X <- as.matrix(mdx$X)
        D <- mdx$D

        message(sprintf("[Info] %s / %s / %s / %s: n=%d, p=%d",
                        method_tag, U_tag, dataset_tag, mode_tag,
                        nrow(X), ncol(X)))

        compute_index_profiles(
          X           = X,
          D           = D,
          dataset_tag = dataset_tag,
          mode_tag    = mode_tag,
          U_tag       = U_tag,
          k_max       = k_max,
          out_csv_dir = out_ds_csv,
          method_tag  = method_tag
        )
      }
    }
  }
}

message("\n✅ Compute finished. CSV profiles are under: ", out_root)



================ METHOD = classical ================



######## U-tag = U10 (k_max = 10) ########


[Info] Reading features: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv (OH)

[Info] OH / top3: selected dim = 3 (method=classical)

[Info] classical / U10 / OH / top3: n=394, p=3


=== [Compute] method=classical, dataset=OH, mode=top3, U=U10 (k=2..10) ===


[DEBUG] classical/OH/top3/U10 k=2 (eff=2), sizes: 17,377

  silhouette = 0.735581

  dunn = 0.244361

  DB = 1.294636

  CH = 108.489557

  ptbiserial = 0.759966


[DEBUG] classical/OH/top3/U10 k=3 (eff=3), sizes: 17,28,349

  silhouette = 0.553750

  dunn = 0.036152

  DB = 1.196267

  CH = 116.329299

  ptbiserial = 0.718457


[DEBUG] classical/OH/top3/U10 k=4 (eff=4), sizes: 2,15,28,349

  silhouette = 0.562313

  dunn = 0.038433

  DB = 0.851067

  CH = 139.081614

  ptbiserial = 0.722882


[DEBUG] cla


[DEBUG] classical/FP/top3/U10 k=3 (eff=3), sizes: 19,52,180

  silhouette = 0.313878

  dunn = 0.050018

  DB = 1.069663

  CH = 70.713763

  ptbiserial = 0.426001


[DEBUG] classical/FP/top3/U10 k=4 (eff=4), sizes: 19,19,52,161

  silhouette = 0.346270

  dunn = 0.051684

  DB = 0.959043

  CH = 83.151648

  ptbiserial = 0.527284


[DEBUG] classical/FP/top3/U10 k=5 (eff=5), sizes: 19,19,24,52,137

  silhouette = 0.360711

  dunn = 0.051684

  DB = 0.895575

  CH = 99.579604

  ptbiserial = 0.561353


[DEBUG] classical/FP/top3/U10 k=6 (eff=6), sizes: 7,19,19,24,52,130

  silhouette = 0.394715

  dunn = 0.053488

  DB = 0.772693

  CH = 115.010290

  ptbiserial = 0.626428


[DEBUG] classical/FP/top3/U10 k=7 (eff=7), sizes: 7,14,19,19,24,38,130

  silhouette = 0.432314

  dunn = 0.071591

  DB = 0.623662

  CH = 143.647987

  ptbiserial = 0.647233


[DEBUG] classical/FP/top3/U10 k=8 (eff=8), sizes: 7,14,19,19,24,38,40,90

  silhouette = 0.426482

  dunn = 0.081671

  DB = 0.773597

  CH

  silhouette = 0.448012

  dunn = 0.067088

  DB = 0.711875

  CH = 197.771465

  ptbiserial = 0.526086


[DEBUG] classical/OH/top3/U25 k=7 (eff=7), sizes: 2,7,8,28,42,86,221

  silhouette = 0.511457

  dunn = 0.041109

  DB = 0.678269

  CH = 251.230188

  ptbiserial = 0.543446


[DEBUG] classical/OH/top3/U25 k=8 (eff=8), sizes: 2,7,8,11,17,42,86,221

  silhouette = 0.527131

  dunn = 0.041109

  DB = 0.647404

  CH = 260.551636

  ptbiserial = 0.545757


[DEBUG] classical/OH/top3/U25 k=9 (eff=9), sizes: 2,2,6,7,11,17,42,86,221

  silhouette = 0.530892

  dunn = 0.048353

  DB = 0.621337

  CH = 276.593515

  ptbiserial = 0.546402


[DEBUG] classical/OH/top3/U25 k=10 (eff=10), sizes: 2,2,3,3,7,11,17,42,86,221

  silhouette = 0.531408

  dunn = 0.049490

  DB = 0.596646

  CH = 274.930584

  ptbiserial = 0.546686


[DEBUG] classical/OH/top3/U25 k=11 (eff=11), sizes: 1,2,2,3,3,6,11,17,42,86,221

  silhouette = 0.531311

  dunn = 0.083790

  DB = 0.570050

  CH = 278.280995

  ptbiserial


[DEBUG] classical/OH/cumeig/U25 k=14 (eff=14), sizes: 2,2,2,3,3,3,4,5,5,7,7,10,11,330

  silhouette = 0.156536

  dunn = 0.206921

  DB = 1.223631

  CH = 7.844870

  ptbiserial = 0.369224


[DEBUG] classical/OH/cumeig/U25 k=15 (eff=15), sizes: 2,2,2,3,3,3,3,4,4,5,5,7,10,11,330

  silhouette = 0.165339

  dunn = 0.206921

  DB = 1.133537

  CH = 7.943278

  ptbiserial = 0.369542


[DEBUG] classical/OH/cumeig/U25 k=16 (eff=16), sizes: 2,2,2,3,3,3,3,4,4,5,5,5,7,10,11,325

  silhouette = 0.171917

  dunn = 0.206921

  DB = 1.173910

  CH = 8.049311

  ptbiserial = 0.405071


[DEBUG] classical/OH/cumeig/U25 k=17 (eff=17), sizes: 2,2,2,3,3,3,3,4,4,5,5,5,7,10,11,35,290

  silhouette = 0.179286

  dunn = 0.196912

  DB = 1.605595

  CH = 8.145968

  ptbiserial = 0.443460


[DEBUG] classical/OH/cumeig/U25 k=18 (eff=18), sizes: 2,2,2,3,3,3,3,4,4,5,5,5,6,7,10,11,29,290

  silhouette = 0.188774

  dunn = 0.196912

  DB = 1.561203

  CH = 8.247596

  ptbiserial = 0.445717


[DEBUG] classical/OH/c

  CH = 220.549809

  ptbiserial = 0.432439


[DEBUG] classical/FP/top3/U25 k=21 (eff=21), sizes: 1,1,3,5,6,7,7,9,9,9,11,11,12,12,13,14,18,19,22,24,38

  silhouette = 0.471063

  dunn = 0.066931

  DB = 0.661162

  CH = 222.636471

  ptbiserial = 0.425846


[DEBUG] classical/FP/top3/U25 k=22 (eff=22), sizes: 1,1,3,5,6,6,6,7,7,9,9,9,11,11,12,13,14,18,19,22,24,38

  silhouette = 0.471617

  dunn = 0.066931

  DB = 0.686961

  CH = 225.308945

  ptbiserial = 0.424212


[DEBUG] classical/FP/top3/U25 k=23 (eff=23), sizes: 1,1,3,5,6,6,6,7,7,8,9,9,9,11,11,12,13,14,18,19,22,24,30

  silhouette = 0.483506

  dunn = 0.066931

  DB = 0.690085

  CH = 228.328188

  ptbiserial = 0.404172


[DEBUG] classical/FP/top3/U25 k=24 (eff=24), sizes: 1,1,3,5,6,6,6,7,7,8,9,9,9,9,10,11,11,12,13,14,18,22,24,30

  silhouette = 0.465734

  dunn = 0.091528

  DB = 0.738438

  CH = 232.292862

  ptbiserial = 0.398929


[DEBUG] classical/FP/top3/U25 k=25 (eff=25), sizes: 1,1,3,3,5,6,6,6,7,7,8,8,9,9,9,9,10,11,12,13,14

...

  [Info] isoMDS final stress: 38.5288

[Saved isoMDS stress] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/OH_FP_only/isoMDS_stress/isoMDS_stress_FP_top3_nonmetric_20251128_115511.csv

[Info] nonmetric / U50 / FP / top3: n=251, p=3


=== [Compute] method=nonmetric, dataset=FP, mode=top3, U=U50 (k=2..50) ===


[DEBUG] nonmetric/FP/top3/U50 k=2 (eff=2), sizes: 71,180

  silhouette = 0.272009

  dunn = 0.045827

  DB = 1.745282

  CH = 59.018521

  ptbiserial = 0.364176


[DEBUG] nonmetric/FP/top3/U50 k=3 (eff=3), sizes: 19,52,180

  silhouette = 0.313878

  dunn = 0.050018

  DB = 1.069663

  CH = 70.713763

  ptbiserial = 0.426001


[DEBUG] nonmetric/FP/top3/U50 k=4 (eff=4), sizes: 19,19,52,161

  silhouette = 0.346270

  dunn = 0.051684

  DB = 0.959043

  CH = 83.151648

  ptbiserial = 0.527284


[DEBUG] nonmetric/FP/top3/U50 k=5 (eff=5), sizes: 19,19,24,52,137

  silhouette = 0.360711

  d

  silhouette = 0.508551

  dunn = 0.159897

  DB = 0.595190

  CH = 298.719831

  ptbiserial = 0.371244


[DEBUG] nonmetric/FP/top3/U50 k=40 (eff=40), sizes: 1,1,1,1,1,2,2,2,3,3,3,3,3,3,3,3,3,3,3,5,5,5,6,6,6,7,7,7,8,8,9,10,11,11,11,12,15,18,19,21

  silhouette = 0.496917

  dunn = 0.080764

  DB = 0.615026

  CH = 304.238043

  ptbiserial = 0.344355


[DEBUG] nonmetric/FP/top3/U50 k=41 (eff=41), sizes: 1,1,1,1,1,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,5,5,5,5,6,6,6,7,7,8,8,9,10,11,11,11,12,15,18,19,21

  silhouette = 0.500637

  dunn = 0.080764

  DB = 0.601381

  CH = 309.740475

  ptbiserial = 0.343429


[DEBUG] nonmetric/FP/top3/U50 k=42 (eff=42), sizes: 1,1,1,1,1,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,5,5,5,5,6,6,6,7,7,8,8,9,9,10,11,11,11,15,18,19,21

  silhouette = 0.502339

  dunn = 0.080764

  DB = 0.608041

  CH = 315.735346

  ptbiserial = 0.340727


[DEBUG] nonmetric/FP/top3/U50 k=43 (eff=43), sizes: 1,1,1,1,1,1,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,5,5,5,5,6,6,6,7,7,8,8,9,9,10,11,11,11,15,18,19,21

initial  value 10.208135 
iter   5 value 3.006984
iter  10 value 2.020926
iter  15 value 1.746345
iter  20 value 1.633562
iter  25 value 1.584047
iter  30 value 1.560752
iter  35 value 1.544376
iter  35 value 1.542970
iter  35 value 1.542488
final  value 1.542488 
converged


  [Info] isoMDS final stress: 1.5425

[Saved isoMDS stress] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/OH_FP_only/isoMDS_stress/isoMDS_stress_FP_cumeig_nonmetric_20251128_115511.csv

[Info] nonmetric / U50 / FP / cumeig: n=251, p=50


=== [Compute] method=nonmetric, dataset=FP, mode=cumeig, U=U50 (k=2..50) ===


[DEBUG] nonmetric/FP/cumeig/U50 k=2 (eff=2), sizes: 28,223

  silhouette = -0.043423

  dunn = 0.283345

  DB = 3.241370

  CH = 6.386499

  ptbiserial = -0.140379


[DEBUG] nonmetric/FP/cumeig/U50 k=3 (eff=3), sizes: 28,59,164

  silhouette = -0.028214

  dunn = 0.280810

  DB = 4.354907

  CH = 6.318988

  ptbiserial = -0.156860


[DEBUG] nonmetric/FP/cumeig/U50 k=4 (eff=4), sizes: 25,28,34,164

  silhouette = -0.026655

  dunn = 0.280810

  DB = 3.639837

  CH = 6.231650

  ptbiserial = -0.194737


[DEBUG] nonmetric/FP/cumeig/U50 k=5 (eff=5), sizes: 19,25,28,34,145

  silhouett

  silhouette = 0.162963

  dunn = 0.284264

  DB = 1.726839

  CH = 6.076074

  ptbiserial = 0.347092


[DEBUG] nonmetric/FP/cumeig/U50 k=40 (eff=40), sizes: 2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,5,5,5,5,5,6,7,7,8,11,12,14,16,17,20,21,21

  silhouette = 0.167393

  dunn = 0.284264

  DB = 1.706915

  CH = 6.108408

  ptbiserial = 0.348720


[DEBUG] nonmetric/FP/cumeig/U50 k=41 (eff=41), sizes: 2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,5,5,5,5,5,6,7,7,8,9,11,12,12,14,16,17,20,21

  silhouette = 0.174254

  dunn = 0.284264

  DB = 1.708557

  CH = 6.138163

  ptbiserial = 0.344437


[DEBUG] nonmetric/FP/cumeig/U50 k=42 (eff=42), sizes: 2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,5,5,5,5,5,6,6,7,7,8,9,11,11,12,12,14,16,20,21

  silhouette = 0.171577

  dunn = 0.284264

  DB = 1.705533

  CH = 6.171978

  ptbiserial = 0.344730


[DEBUG] nonmetric/FP/cumeig/U50 k=43 (eff=43), sizes: 2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,5,5,5,5,5,5,6,6,7,7,9,11,11,12,12,14,16,20,21

